# Hitung ISPU

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FILE = ROOT / "data" / "Dataset GelarRasa Bismillah aja (3).xlsx"
OUT = ROOT / "hasil_ispu"
if not FILE.is_file():
    raise FileNotFoundError(f"File belum ada: {FILE}")
OUT.mkdir(parents=True, exist_ok=True)


## Baca konsentrasi harian

In [ ]:
df = pd.read_excel(FILE, sheet_name="03_AQMS_Stasiun")
ids = ["tanggal", "sensor_id", "provinsi", "kota_kabupaten"]
polutan = {
    "PM2.5": "pm25_ug_m3", "PM10": "pm10_ug_m3", "SO2": "so2_ug_m3",
    "CO": "co_ug_m3", "O3": "o3_ug_m3", "NO2": "no2_ug_m3"
}
cols = ids + list(polutan.values())
assert set(cols).issubset(df.columns), "Kolom sumber tidak lengkap"
data = df[cols].copy()
data["tanggal"] = pd.to_datetime(data["tanggal"], errors="coerce")
for c in polutan.values():
    data[c] = pd.to_numeric(data[c], errors="coerce")
print("Stasiun-hari:", len(data), "| Sensor:", data.sensor_id.nunique())


## Hitung subindeks

In [ ]:
titik = [0, 50, 100, 200, 300]
batas = {
    "PM2.5": [0, 15.5, 55.4, 150.4, 250.4],
    "PM10": [0, 50, 150, 350, 420],
    "SO2": [0, 52, 180, 400, 800],
    "CO": [0, 4000, 8000, 15000, 30000],
    "O3": [0, 120, 235, 400, 800],
    "NO2": [0, 80, 200, 1130, 2260],
}

def subindeks(value, nama):
    if pd.isna(value) or value < 0:
        return np.nan
    if value > batas[nama][-1]:
        return 301  # Penanda kategori, bukan indeks eksak di atas 300.
    for i in range(1, len(titik)):
        if value <= batas[nama][i]:
            x0, x1 = batas[nama][i-1:i+1]
            return int(round(titik[i-1] + (value-x0)*(titik[i]-titik[i-1])/(x1-x0)))
    return 0

nama_kolom = {k: "ispu_" + k.lower().replace(".", "") for k in polutan}
for nama, src in polutan.items():
    data[nama_kolom[nama]] = data[src].map(lambda v: subindeks(v, nama))
indeks_cols = list(nama_kolom.values())


## Tentukan kategori

In [ ]:
data["ispu_max"] = data[indeks_cols].max(axis=1)

def kritis(row):
    if pd.isna(row.ispu_max):
        return np.nan
    return " / ".join(k for k, v in nama_kolom.items() if row[v] == row.ispu_max)

def kategori(v):
    if pd.isna(v): return np.nan
    if v <= 50: return "Baik"
    if v <= 100: return "Sedang"
    if v <= 200: return "Tidak Sehat"
    if v <= 300: return "Sangat Tidak Sehat"
    return "Berbahaya"

data["parameter_kritis"] = data.apply(kritis, axis=1)
data["kategori_ispu"] = data.ispu_max.map(kategori)
print(data.kategori_ispu.value_counts())


## Simpan hasil

In [ ]:
harian = data[ids + list(polutan.values()) + indeks_cols + ["ispu_max", "parameter_kritis", "kategori_ispu"]].copy()
ringkasan = data.copy()
ringkasan["bulan"] = ringkasan.tanggal.dt.to_period("M").dt.to_timestamp()

def modus(x):
    x = x.dropna()
    return x.mode().iloc[0] if not x.empty else np.nan

bulanan = (ringkasan.groupby(["bulan", "provinsi"], as_index=False)
    .agg(rata_rata_ispu_stasiun=("ispu_max", "mean"),
         ispu_tertinggi_bulanan=("ispu_max", "max"),
         jumlah_pengamatan_stasiun_harian=("ispu_max", "count"),
         parameter_kritis_tersering=("parameter_kritis", modus)))
bulanan["rata_rata_ispu_stasiun"] = bulanan["rata_rata_ispu_stasiun"].round(1)
harian.to_csv(OUT / "ispu_harian.csv", index=False)
bulanan.to_csv(OUT / "ispu_bulanan_provinsi.csv", index=False)
print("Tersimpan:", OUT)


## Cocokkan dengan dataset akhir

In [ ]:
# Audit perbedaan antara hasil hitung ulang dan sheet ISPU akhir.
final = pd.read_excel(FILE, sheet_name="10_ISPU_Harian")
keys = ["tanggal", "sensor_id"]
final["tanggal"] = pd.to_datetime(final.tanggal, errors="coerce")
cek = harian.merge(final[keys + ["ispu_max"]], on=keys,
                   suffixes=("_hitung", "_akhir"), how="outer", indicator=True)
cek["selisih"] = cek.ispu_max_hitung - cek.ispu_max_akhir
print("Kecocokan tanggal-sensor:", cek._merge.value_counts().to_dict())
print("Selisih ISPU pada baris cocok:", int(cek.selisih.abs().gt(0.01).sum()))
print("Selisih maksimum:", cek.selisih.abs().max())
cek.loc[(cek._merge != "both") | cek.selisih.abs().gt(0.01)].to_csv(
    OUT / "audit_perbedaan_ispu.csv", index=False)
print("Lihat audit_perbedaan_ispu.csv jika hasil berbeda.")
